# 06 — LLM-Pipeline 3: Tool-Use (aktive Datenabfrage)

**Idee:** Das LLM erhält nur den Kontext der Aufgabe und eine Beschreibung der verfügbaren Tools. Es entscheidet **selbst**, welche Daten es abruft,
und kann iterativ mehrere Tools aufrufen, bevor es eine Erklärung formuliert.

**Verfügbare Tools:**
| Tool | Beschreibung |
|------|--------------|
| `get_feature_schema` | Metadaten zu allen Features |
| `get_feature_importance` | Globale Feature-Wichtigkeit |
| `get_prediction` | Vorhersage für eigene Feature-Kombination |
| `get_shap_values` | Lokale Beiträge für eine Testinstanz |
| `get_partial_dependence` | Partial-Dependence-Kurve für ein Feature |
| `get_feature_value_context` | Einordnung eines Featurewerts im Trainingsset (Percentile) |
| `get_similar_instances` | k ähnlichste Trainingsinstanzen |
| `get_counterfactual_prediction` | Was-wäre-wenn-Vorhersage bei geänderten Features |


In [1]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib

from utils import (
    INSTANCE_IDS, N_GENERATIONS_SCALE, scale_instance_ids,
    MODELS_DIR, RESULTS_DIR, PROMPTS_DIR,
)
from utils.data import load_train_test
from utils.tools import ToolBox, TOOL_DEFINITIONS
from utils.llm import DEFAULT_MODEL, _get_client, _with_retry, strip_scratchpad

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = 2048

# ── Phase 3b — Skalierungsschalter ────────────────────────────────────────────
# Tool-Use ist NICHT batchbar (client-seitiger Tool-Loop) → bleibt real-time, auch
# bei n≈200. SCALE_RUN wählt Instanzmenge, Generationenzahl und Ausgabeordner
# (results/pipeline06/scale/ — physisch getrennt von der n=20-Validität).
SCALE_RUN = True
if SCALE_RUN:
    GEN_INSTANCE_IDS = scale_instance_ids()
    N_GEN            = N_GENERATIONS_SCALE
    OUT_DIR          = RESULTS_DIR / 'pipeline06' / 'scale'
else:
    GEN_INSTANCE_IDS = INSTANCE_IDS
    N_GEN            = 1
    OUT_DIR          = RESULTS_DIR / 'pipeline06'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'LLM-Modell:    {MODEL}')
print(f'SCALE_RUN:     {SCALE_RUN}  (n={len(GEN_INSTANCE_IDS)} Instanzen × {N_GEN} Generationen, real-time)')
print(f'Ausgabe:       {OUT_DIR}')

LLM-Modell:    claude-sonnet-4-6
SCALE_RUN:     True  (n=5 Instanzen × 3 Generationen, real-time)
Ausgabe:       /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/pipeline06/scale


## 1 · Daten und Modelle laden

In [2]:
X_train, y_train, X_test, y_test = load_train_test()
from utils.models import load_models
xgb, ebm = load_models(LOSS_KEY)
print(f'X_test: {X_test.shape} | Tools: {[t["name"] for t in TOOL_DEFINITIONS]}')

X_test: (5227, 9) | Tools: ['get_feature_schema', 'get_feature_importance', 'get_prediction', 'get_shap_values', 'get_partial_dependence', 'get_feature_value_context', 'get_similar_instances', 'get_counterfactual_prediction']


## 2 · System-Prompt und Tool-Use-Schleife

In [3]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_06_tooluse.md").read_text()


def tool_use_loop(
    client,
    toolbox: "ToolBox",
    user_message: str,
    system: str,
    max_rounds: int = 10,
) -> tuple[str, list[dict], int, int, str]:
    """
    Führt die Tool-Use-Schleife aus.
    Gibt zurück: (finaler Text, tool_calls_log, input_tokens, output_tokens, stop_reason)
    stop_reason: 'end_turn' | 'max_tokens' | 'max_rounds' | anderer API-Grund
    """
    messages = [{"role": "user", "content": user_message}]
    total_in, total_out = 0, 0
    last_text = ""
    final_stop_reason = "max_rounds"

    for round_num in range(max_rounds):
        response = _with_retry(
            client.messages.create,
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=[
                {"type": "text", "text": system,
                 "cache_control": {"type": "ephemeral"}}
            ],
            tools=TOOL_DEFINITIONS,
            messages=messages,
        )
        usage = response.usage
        total_in  += usage.input_tokens
        total_out += usage.output_tokens

        messages.append({"role": "assistant", "content": response.content})

        candidate = next(
            (b.text for b in response.content if hasattr(b, "text") and b.text), ""
        )
        if candidate:
            last_text = candidate

        if response.stop_reason == "end_turn":
            return last_text, toolbox.call_log, total_in, total_out, "end_turn"

        if response.stop_reason != "tool_use":
            final_stop_reason = response.stop_reason
            break

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            result = toolbox.dispatch(block.name, block.input)
            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     json.dumps(result, ensure_ascii=False),
            })
            print(f"    [{round_num+1}] {block.name}({list(block.input.keys())}) -> ok")

        messages.append({"role": "user", "content": tool_results})

    if final_stop_reason == "max_rounds":
        print(f"  [WARN] max_rounds={max_rounds} erreicht – verwende letzten Teiltext.")
    else:
        print(f"  [WARN] stop_reason='{final_stop_reason}' – verwende letzten Teiltext.")
    return last_text or "[Keine Antwort]", toolbox.call_log, total_in, total_out, final_stop_reason


print("Tool-Use-Schleife definiert.")


Tool-Use-Schleife definiert.


## 3 · Aufgaben-Prompt pro Instanz


In [4]:
from utils.explanations import (
    WEEKDAY_NAMES, MONTH_NAMES, WEATHER_NAMES,
    TEMP_FACTOR, HUM_FACTOR, WIND_FACTOR,
)


def build_task_prompt(model_name: str, instance_id: int) -> str:
    row = X_test.iloc[instance_id]
    y   = float(y_test.iloc[instance_id])

    temp_c   = float(row["temp"])      * TEMP_FACTOR
    hum_pct  = float(row["hum"])       * HUM_FACTOR
    wind_kmh = float(row["windspeed"]) * WIND_FACTOR

    lines = [
        f"Ich möchte die Vorhersage für Testinstanz {instance_id} verstehen.",
        f"",
        f"Situation (denormalisierte Werte zur Orientierung):",
        f"  Modell:              {model_name.upper()}",
        f"  Uhrzeit:             {int(row['hr']):02d}:00 Uhr",
        f"  Wochentag:           {WEEKDAY_NAMES[int(row['weekday'])]}",
        f"  Monat:               {MONTH_NAMES[int(row['mnth'])]}",
        f"  Jahr:                {'2012' if int(row['yr']) == 1 else '2011'}",
        f"  Wetter:              {WEATHER_NAMES.get(int(row['weathersit']), int(row['weathersit']))}",
        f"  Temperatur:          ~{temp_c:.1f} °C",
        f"  Luftfeuchtigkeit:    {hum_pct:.0f} %",
        f"  Windgeschwindigkeit: {wind_kmh:.1f} km/h",
        f"  Feiertag:            {'ja' if int(row['holiday']) == 1 else 'nein'}",
        f"  Tatsächliche Ausleihe: {int(y)} Fahrräder",
        f"",
        f"Bitte nutze mindestens 4 der verfügbaren Tools, um die Vorhersage",
        f"umfassend zu analysieren. Rufe insbesondere get_shap_values und",
        f"get_feature_value_context für die wichtigsten Treiber auf.",
        f"Formuliere anschließend eine verständliche Erklärung auf Deutsch.",
    ]
    return "\n".join(lines)


# Beispiel-Prompt
print(build_task_prompt("xgb", INSTANCE_IDS[0]))


Ich möchte die Vorhersage für Testinstanz 224 verstehen.

Situation (denormalisierte Werte zur Orientierung):
  Modell:              XGB
  Uhrzeit:             19:00 Uhr
  Wochentag:           Donnerstag
  Monat:               Februar
  Jahr:                2011
  Wetter:              klar/wenige Wolken
  Temperatur:          ~8.2 °C
  Luftfeuchtigkeit:    40 %
  Windgeschwindigkeit: 0.0 km/h
  Feiertag:            nein
  Tatsächliche Ausleihe: 96 Fahrräder

Bitte nutze mindestens 4 der verfügbaren Tools, um die Vorhersage
umfassend zu analysieren. Rufe insbesondere get_shap_values und
get_feature_value_context für die wichtigsten Treiber auf.
Formuliere anschließend eine verständliche Erklärung auf Deutsch.


## 4 · LLM-Aufrufe

> **Voraussetzung:** `ANTHROPIC_API_KEY` in `.env` oder als Umgebungsvariable.

In [5]:
from utils import run_resumable_generation

client  = _get_client()
model_objs = {"xgb": xgb, "ebm": ebm}

# Resume/Persistenz-Kontrakt zentral in utils.run_resumable_generation
# (skip-if-exists, idempotent, verlustfrei — getestet in tests/test_generation_loop.py).
# generate gibt bei einem Fehler None zurück → Instanz bleibt offen und wird
# beim nächsten Lauf erneut versucht (statt fehlerhaft persistiert zu werden).
# Bei N_GEN > 1 wird jede Instanz N-mal gerufen (gen_idx 0..N-1) → eigene Dateien
# (…_gen{idx}.json). Eine frische ToolBox je Aufruf → eigener call_log je Generation.
totals = {"in": 0, "out": 0}


def generate_tooluse(model_name, iid, gen_idx):
    toolbox = ToolBox(model_objs[model_name], X_train, X_test, y_test, model_name)
    prompt  = build_task_prompt(model_name, iid)
    system  = SYSTEM_PROMPT.replace('{model_name}', model_name.upper())

    print(f'\n{model_name.upper()} inst={iid} g{gen_idx}')
    t0 = time.time()
    try:
        raw_text, call_log, in_tok, out_tok, stop_reason = tool_use_loop(
            client, toolbox, prompt, system
        )
    except Exception as e:
        print(f'  [ERROR] {type(e).__name__}: {e} – überspringe Instanz.')
        return None
    elapsed = time.time() - t0
    text = strip_scratchpad(raw_text)
    totals["in"] += in_tok; totals["out"] += out_tok

    print(f'  -> {len(call_log)} Tool-Aufrufe  '
          f'in={in_tok}  out={out_tok}  t={elapsed:.1f}s  stop={stop_reason}')

    return {
        'pipeline':    '06_tooluse',
        'llm_model':   MODEL,
        'loss_key':    LOSS_KEY,
        'xai_model':   model_name,
        'instance_id': iid,
        'explanation': text,
        'stop_reason': stop_reason,
        'tool_calls':  call_log,
        'n_tool_calls': len(call_log),
        'elapsed_s':   round(elapsed, 2),
        'usage':       {'input_tokens': in_tok, 'output_tokens': out_tok},
        'y_true':      float(y_test.iloc[iid]),
    }


def _on_skip(record, model_name, iid, gen_idx):
    pass  # bei n≈200 nicht je übersprungene Einheit loggen


results = run_resumable_generation(
    model_names=["xgb", "ebm"],
    instance_ids=GEN_INSTANCE_IDS,
    out_dir=OUT_DIR,
    generate=generate_tooluse,
    n_generations=N_GEN,
    on_skip=_on_skip,
)

print(f'\nGesamt:  input={totals["in"]}  output={totals["out"]}  ({len(results)} Einheiten)')


XGB inst=419 g0
    [1] get_shap_values(['instance_id']) -> ok
    [1] get_feature_importance([]) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_counterfactual_prediction(['instance_id', 'changes']) -> ok
  -> 5 Tool-Aufrufe  in=4341  out=1564  t=39.6s  stop=end_turn

XGB inst=419 g1
    [1] get_shap_values(['instance_id']) -> ok
    [1] get_feature_importance([]) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [3] get_counterfactual_prediction(['instance_id', 'changes']) -> ok
  -> 5 Tool-Aufrufe  in=6965  out=1543  t=40.9s  stop=end_turn

XGB inst=419 g2
    [1] get_shap_values(['instance_id']) -> ok
    [1] get_feature_importance([]) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [3]

## 5 · Beispiel-Erklärung und Tool-Trace

In [6]:
rec = results[0]  # erste XGBoost-Erklärung
sep = '=' * 70
print(sep)
print(f"Modell: {rec['xai_model'].upper()}  |  Instanz: {rec['instance_id']}  "
      f"|  Tool-Aufrufe: {rec['n_tool_calls']}")
print(sep)
print(rec['explanation'])
print()
print('Tool-Trace:')
for i, call in enumerate(rec['tool_calls']):
    print(f"  [{i+1}] {call['tool']}({list(call['arguments'].keys())})")
    print(f"       -> {call['result_preview'][:120]}")

Modell: XGB  |  Instanz: 419  |  Tool-Aufrufe: 5
Alle Daten sind vorhanden. Ich erstelle nun die vollständige Analyse.

<vorhersage>
Das Modell sagte für diese Stunde rund 2 ausgeliehene Fahrräder vorher; tatsächlich wurden 4 gezählt. Die Abweichung beträgt etwa 2 Räder — das ist eine moderate Unterschätzung, angesichts der ohnehin extrem niedrigen Nachfrage in dieser Nacht aber als gut getroffen zu bewerten. Bei Werten unter 10 Ausleihungen sind kleine absolute Abweichungen unvermeidlich.
</vorhersage>

<treiber>
Mit Abstand der stärkste dämpfende Faktor ist die Uhrzeit 03:00 Uhr (Einfluss −3,08). Das Modell kennt die Nachtstunden als nahezu nachfragefreie Zeiten, und der Kontextvergleich bestätigt: Die Stunde 3 ist mit rund 4 % aller Trainingsstunden gleichmäßig verteilt — es gibt also keine Besonderheit im Datenvorkommen, sondern schlicht kaum Betrieb zu dieser Zeit. Wie stark dieser Effekt ist, zeigt das Was-wäre-wenn-Szenario eindrucksvoll: Wäre es 08:00 Uhr statt 03:00 Uhr (Morge

## 6 · Zusammenfassung

In [7]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Modell':        r['xai_model'].upper(),
        'Instanz':       r['instance_id'],
        'y_true':        r['y_true'],
        'Tool-Aufrufe':  r['n_tool_calls'],
        'Tools genutzt': ', '.join({c['tool'] for c in r['tool_calls']}),
        'Wörter':        len(r['explanation'].split()),
        'tok_input':     r['usage']['input_tokens'],
        'tok_output':    r['usage']['output_tokens'],
        'Zeit (s)':      r['elapsed_s'],
    }
    for r in results
])
display(summary)

,Modell,Instanz,y_true,Tool-Aufrufe,Tools genutzt,Wörter,tok_input,tok_output,Zeit (s)
0,XGB,419,4.0,5,"get_feature_value_context, get_counterfactual_...",352,4341,1564,39.55
1,XGB,419,4.0,5,"get_feature_value_context, get_counterfactual_...",351,6965,1543,40.86
2,XGB,419,4.0,6,"get_feature_value_context, get_counterfactual_...",335,6736,1601,41.57
3,XGB,3608,228.0,7,"get_counterfactual_prediction, get_feature_imp...",339,7119,1809,44.98
4,XGB,3608,228.0,7,"get_feature_value_context, get_counterfactual_...",355,6601,1808,42.67
5,XGB,3608,228.0,7,"get_counterfactual_prediction, get_similar_ins...",380,7998,2080,47.80
6,XGB,3658,324.0,7,"get_feature_value_context, get_counterfactual_...",374,7164,1797,44.02
7,XGB,3658,324.0,5,"get_feature_value_context, get_counterfactual_...",322,4327,1503,38.45
8,XGB,3658,324.0,5,"get_feature_value_context, get_counterfactual_...",325,4580,1752,40.20
9,XGB,3687,338.0,5,"get_feature_value_context, get_counterfactual_...",311,4297,1397,35.53
